In [1]:
import os
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ACTIVE_PRESET = "standard"

PRESETS = {
    "fast": {
        "quick_test": True,
        "max_train_films": 300,
        "max_test_films": 100,
        "max_train_batches": 120,
        "model_name": "distilbert-base-uncased",
        "max_len": 160,
        "batch_size": 8,
        "epochs": 1,
        "lr": 2e-4,
        "freeze_encoder": True,
    },
    "standard": {
        "quick_test": True,
        "max_train_films": 1500,
        "max_test_films": 300,
        "max_train_batches": None,
        "model_name": "distilbert-base-uncased",
        "max_len": 192,
        "batch_size": 16,
        "epochs": 2,
        "lr": 1e-4,
        "freeze_encoder": True,
    },
    "finetune": {
        "quick_test": False,
        "max_train_films": None,
        "max_test_films": None,
        "max_train_batches": None,
        "model_name": "distilbert-base-uncased",
        "max_len": 192,
        "batch_size": 8,
        "epochs": 2,
        "lr": 2e-5,
        "freeze_encoder": False,
    },
}

CFG = PRESETS[ACTIVE_PRESET]

RANDOM_STATE = 42
MIN_FILMS_PER_COUNTRY = 50

USE_XPU_IF_AVAILABLE = True

if USE_XPU_IF_AVAILABLE and hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
else:
    DEVICE = torch.device("cpu")

print("Active preset:", ACTIVE_PRESET)
print("Torch version:", torch.__version__)
print("Using device:", DEVICE)

if DEVICE.type == "xpu":
    print("XPU device:", torch.xpu.get_device_name(0))

Active preset: standard
Torch version: 2.11.0+xpu
Using device: xpu
XPU device: Intel(R) Arc(TM) 140V GPU (16GB)


In [3]:
import random

MANUAL_PROJECT_ROOT = None

EXPECTED_FILES = [
    "Lumiere_Country_Full_Backup.csv",
    "clustered_films_k6.csv",
    "IMDB_financials.xlsx",
]


def find_project_root(start_path):
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        data_dir = candidate / "analysis" / "data"
        if all((data_dir / file_name).exists() for file_name in EXPECTED_FILES):
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Set MANUAL_PROJECT_ROOT manually."
    )


def create_unique_run_output_dir(base_output_dir, preset_name):
    while True:
        run_number = random.randint(1000, 9999)
        run_name = f"{preset_name}{run_number}"
        run_output_dir = base_output_dir / run_name

        if not run_output_dir.exists():
            run_output_dir.mkdir(parents=True, exist_ok=False)
            return run_output_dir, run_name, run_number


if MANUAL_PROJECT_ROOT is None:
    PROJECT_ROOT = find_project_root(Path.cwd())
else:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "analysis" / "data"

ADMISSIONS_PATH = DATA_DIR / "Lumiere_Country_Full_Backup.csv"
CLUSTERS_PATH = DATA_DIR / "clustered_films_k6.csv"
FINANCIALS_PATH = DATA_DIR / "IMDB_financials.xlsx"

BASE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "distilbert_regressor_xpu"
OUTPUT_DIR, RUN_NAME, RUN_NUMBER = create_unique_run_output_dir(
    BASE_OUTPUT_DIR,
    ACTIVE_PRESET
)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Base output dir:", BASE_OUTPUT_DIR)
print("Run output dir:", OUTPUT_DIR)
print("Run name:", RUN_NAME)

Project root: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action
Data dir: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\analysis\data
Base output dir: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu
Run output dir: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu\standard6511
Run name: standard6511


In [4]:
df_admissions = pd.read_csv(ADMISSIONS_PATH)
df_clusters = pd.read_csv(CLUSTERS_PATH)

budget_both = pd.read_excel(FINANCIALS_PATH, sheet_name="Both")
budget_only = pd.read_excel(FINANCIALS_PATH, sheet_name="Budget only")

df_budget = pd.concat([budget_both, budget_only], ignore_index=True)
df_budget = df_budget[["titleId", "budget_eur"]].copy()
df_budget["budget_eur"] = pd.to_numeric(df_budget["budget_eur"], errors="coerce")
df_budget = df_budget.dropna(subset=["budget_eur"])
df_budget = df_budget.drop_duplicates(subset=["titleId"])

if "titleId" in df_clusters.columns:
    df_clusters_small = df_clusters[["titleId", "cluster"]].drop_duplicates(subset=["titleId"])
    df = df_admissions.merge(df_clusters_small, on="titleId", how="inner")

else:
    if len(df_admissions) != len(df_clusters):
        raise ValueError(
            "The clusters file has no titleId and does not have the same number of rows as the admissions file. "
            "You need to regenerate the clustered CSV with titleId included."
        )

    df_clusters_small = pd.DataFrame({
        "titleId": df_admissions["titleId"].values,
        "cluster": df_clusters["cluster"].values
    })

    df = df_admissions.merge(df_clusters_small, on="titleId", how="inner")

df = df.merge(df_budget, on="titleId", how="left")

print("Admissions rows:", len(df_admissions))
print("Cluster rows:", len(df_clusters))
print("Merged rows:", len(df))
print("Films with missing budget:", df["budget_eur"].isna().sum())

if "titleId" not in df_clusters.columns:
    print("Warning: cluster file had no titleId. Clusters were aligned using row order.")

Admissions rows: 2000
Cluster rows: 2000
Merged rows: 2000
Films with missing budget: 893


In [5]:
def parse_country_admissions(value):
    if pd.isna(value) or value == "Not Found":
        return {}

    if isinstance(value, dict):
        return value

    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, dict):
            return parsed
    except (ValueError, SyntaxError):
        pass

    try:
        parsed = json.loads(str(value).replace("'", '"'))
        if isinstance(parsed, dict):
            return parsed
    except (json.JSONDecodeError, TypeError):
        pass

    return {}


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).replace("\n", " ").replace("\r", " ").strip()


def parse_admission_count(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace(",", "").strip()
    return pd.to_numeric(value, errors="coerce")


def make_budget_bracket(value):
    if pd.isna(value):
        return "unknown"

    if value < 3_000_000:
        return "micro"
    if value < 8_000_000:
        return "european"
    if value < 25_000_000:
        return "mid"
    return "big"


def is_domestic_market(origin_country, market_country):
    origin = str(origin_country).lower()
    market = str(market_country).lower()
    return int(market in origin)

In [6]:
records = []

for _, row in df.iterrows():
    country_admissions = parse_country_admissions(row["country_admissions"])

    for country, admissions in country_admissions.items():
        if str(country).startswith("Total"):
            continue

        records.append({
            "titleId": row["titleId"],
            "originalTitle": row.get("originalTitle", ""),
            "plotMedium": row.get("plotMedium", ""),
            "mainCountry": row.get("mainCountry", ""),
            "firstLanguage": row.get("firstLanguage", ""),
            "genres": row.get("genres", ""),
            "keywords": row.get("keywords", ""),
            "cluster": row.get("cluster", np.nan),
            "budget_eur": row.get("budget_eur", np.nan),
            "country": country,
            "admissions": parse_admission_count(admissions),
        })

df_long = pd.DataFrame(records)

df_long = df_long.dropna(subset=["admissions"])
df_long = df_long[df_long["admissions"] >= 0].copy()

print("Long rows:", len(df_long))
print("Unique films:", df_long["titleId"].nunique())
print("Unique countries:", df_long["country"].nunique())

Long rows: 22976
Unique films: 1840
Unique countries: 40


In [7]:
films_with_gbie = set(df_long.loc[df_long["country"] == "GBIE", "titleId"])

mask_redundant = (
    df_long["country"].isin(["GB", "IE"]) &
    df_long["titleId"].isin(films_with_gbie)
)

df_long = df_long.loc[~mask_redundant].copy()

country_film_counts = df_long.groupby("country")["titleId"].nunique()
valid_countries = country_film_counts[country_film_counts >= MIN_FILMS_PER_COUNTRY].index

df_long = df_long[df_long["country"].isin(valid_countries)].copy()

df_long["log_admissions"] = np.log1p(df_long["admissions"])

country_mean = df_long.groupby("country")["log_admissions"].transform("mean")
country_std = df_long.groupby("country")["log_admissions"].transform("std")

df_long["admissions_zscore"] = (df_long["log_admissions"] - country_mean) / country_std

df_long = df_long.replace([np.inf, -np.inf], np.nan)
df_long = df_long.dropna(subset=["admissions_zscore"])

print("Rows after filtering:", len(df_long))
print("Films after filtering:", df_long["titleId"].nunique())
print("Countries after filtering:", df_long["country"].nunique())

Rows after filtering: 22900
Films after filtering: 1840
Countries after filtering: 35


In [8]:
df_long["budget_bracket"] = df_long["budget_eur"].apply(make_budget_bracket)
df_long["has_budget"] = df_long["budget_eur"].notna().astype(int)
df_long["log_budget_eur"] = np.log1p(df_long["budget_eur"])

df_long["plot_clean"] = df_long["plotMedium"].apply(clean_text)
df_long["plot_word_count"] = df_long["plot_clean"].str.split().str.len()
df_long["log_plot_word_count"] = np.log1p(df_long["plot_word_count"])

df_long["is_domestic"] = df_long.apply(
    lambda row: is_domestic_market(row["mainCountry"], row["country"]),
    axis=1
)

df_long["model_text"] = (
    "Market: " + df_long["country"].astype(str) +
    " | Origin: " + df_long["mainCountry"].astype(str) +
    " | Language: " + df_long["firstLanguage"].astype(str) +
    " | Genres: " + df_long["genres"].astype(str) +
    " | Keywords: " + df_long["keywords"].astype(str) +
    " | Cluster: " + df_long["cluster"].astype(str) +
    " | Budget bracket: " + df_long["budget_bracket"].astype(str) +
    " | Plot: " + df_long["plot_clean"]
)

df_long = df_long[df_long["model_text"].str.len() > 30].copy()

NUMERIC_COLUMNS = [
    "log_budget_eur",
    "has_budget",
    "is_domestic",
    "log_plot_word_count",
]

print(df_long[["titleId", "country", "admissions_zscore", "model_text"]].head(2))
print("Numeric columns:", NUMERIC_COLUMNS)

     titleId country  admissions_zscore  \
0  tt3042384      AT          -0.980048   
1  tt3042384      BE           0.120957   

                                          model_text  
0  Market: AT | Origin: PL | Language: pl | Genre...  
1  Market: BE | Origin: PL | Language: pl | Genre...  
Numeric columns: ['log_budget_eur', 'has_budget', 'is_domestic', 'log_plot_word_count']


In [9]:
film_ids = df_long["titleId"].drop_duplicates().values

train_films, test_films = train_test_split(
    film_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

if CFG["quick_test"]:
    rng = np.random.default_rng(RANDOM_STATE)

    if CFG["max_train_films"] is not None:
        train_films = rng.choice(
            train_films,
            size=min(CFG["max_train_films"], len(train_films)),
            replace=False
        )

    if CFG["max_test_films"] is not None:
        test_films = rng.choice(
            test_films,
            size=min(CFG["max_test_films"], len(test_films)),
            replace=False
        )

train_df = df_long[df_long["titleId"].isin(train_films)].copy()
test_df = df_long[df_long["titleId"].isin(test_films)].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train films:", train_df["titleId"].nunique())
print("Test films:", test_df["titleId"].nunique())

Train rows: 18089
Test rows: 3904
Train films: 1472
Test films: 300


In [10]:
train_medians = train_df[NUMERIC_COLUMNS].median(numeric_only=True)

train_df[NUMERIC_COLUMNS] = train_df[NUMERIC_COLUMNS].fillna(train_medians)
test_df[NUMERIC_COLUMNS] = test_df[NUMERIC_COLUMNS].fillna(train_medians)

feature_means = train_df[NUMERIC_COLUMNS].mean()
feature_stds = train_df[NUMERIC_COLUMNS].std().replace(0, 1)

train_df[NUMERIC_COLUMNS] = (train_df[NUMERIC_COLUMNS] - feature_means) / feature_stds
test_df[NUMERIC_COLUMNS] = (test_df[NUMERIC_COLUMNS] - feature_means) / feature_stds

print("Numeric feature means after scaling:")
print(train_df[NUMERIC_COLUMNS].mean().round(4))

print("\nNumeric feature stds after scaling:")
print(train_df[NUMERIC_COLUMNS].std().round(4))

Numeric feature means after scaling:
log_budget_eur         0.0
has_budget            -0.0
is_domestic            0.0
log_plot_word_count   -0.0
dtype: float64

Numeric feature stds after scaling:
log_budget_eur         1.0
has_budget             1.0
is_domestic            1.0
log_plot_word_count    1.0
dtype: float64


In [11]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])


class FilmAdmissionsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, numeric_columns, max_len):
        self.labels = torch.tensor(
            dataframe["admissions_zscore"].values,
            dtype=torch.float32
        )

        self.numeric_features = torch.tensor(
            dataframe[numeric_columns].values,
            dtype=torch.float32
        )

        self.encodings = tokenizer(
            dataframe["model_text"].tolist(),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": self.encodings["attention_mask"][index],
            "numeric_features": self.numeric_features[index],
            "labels": self.labels[index],
        }
        return item


train_dataset = FilmAdmissionsDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    numeric_columns=NUMERIC_COLUMNS,
    max_len=CFG["max_len"]
)

test_dataset = FilmAdmissionsDataset(
    dataframe=test_df,
    tokenizer=tokenizer,
    numeric_columns=NUMERIC_COLUMNS,
    max_len=CFG["max_len"]
)

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 18089
Test dataset: 3904


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 1131
Test batches: 244


In [13]:
class DistilBertAdmissionsRegressor(torch.nn.Module):
    def __init__(self, model_name, num_numeric_features, freeze_encoder=True):
        super().__init__()

        self.freeze_encoder = freeze_encoder
        self.encoder = AutoModel.from_pretrained(model_name)

        if self.freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.text_layer = torch.nn.Sequential(
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_size, 128),
            torch.nn.ReLU()
        )

        self.numeric_layer = torch.nn.Sequential(
            torch.nn.Linear(num_numeric_features, 32),
            torch.nn.ReLU()
        )

        self.regressor = torch.nn.Sequential(
            torch.nn.Dropout(0.1),
            torch.nn.Linear(160, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(64, 1)
        )

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def encode_text(self, input_ids, attention_mask):
        if self.freeze_encoder:
            self.encoder.eval()
            with torch.no_grad():
                outputs = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
        else:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        return self.mean_pooling(outputs.last_hidden_state, attention_mask)

    def forward(self, input_ids, attention_mask, numeric_features):
        text_embedding = self.encode_text(input_ids, attention_mask)
        text_output = self.text_layer(text_embedding)
        numeric_output = self.numeric_layer(numeric_features)

        combined = torch.cat([text_output, numeric_output], dim=1)
        prediction = self.regressor(combined).squeeze(-1)

        return prediction


model = DistilBertAdmissionsRegressor(
    model_name=CFG["model_name"],
    num_numeric_features=len(NUMERIC_COLUMNS),
    freeze_encoder=CFG["freeze_encoder"]
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Model ready")
print("Device:", DEVICE)
print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready
Device: xpu
Trainable parameters: 108961
Total parameters: 66471841


In [14]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["lr"]
)

loss_fn = torch.nn.MSELoss()

for epoch in range(CFG["epochs"]):
    model.train()
    total_loss = 0.0
    used_batches = 0

    for batch_index, batch in enumerate(train_loader):
        if CFG["max_train_batches"] is not None and batch_index >= CFG["max_train_batches"]:
            break

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        numeric_features = batch["numeric_features"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        preds = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        loss = loss_fn(preds, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        used_batches += 1

    avg_loss = total_loss / max(1, used_batches)

    print(f"Epoch {epoch + 1}/{CFG['epochs']} - Train loss: {avg_loss:.4f} - Batches: {used_batches}")

Epoch 1/2 - Train loss: 0.8769 - Batches: 1131
Epoch 2/2 - Train loss: 0.8283 - Batches: 1131


In [15]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        numeric_features = batch["numeric_features"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        preds = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        all_preds.extend(preds.detach().to("cpu").numpy())
        all_labels.extend(labels.detach().to("cpu").numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

r2 = r2_score(all_labels, all_preds)
rmse = np.sqrt(mean_squared_error(all_labels, all_preds))

baseline_preds = np.zeros_like(all_labels)
baseline_r2 = r2_score(all_labels, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(all_labels, baseline_preds))

print(f"Model Test R2: {r2:.4f}")
print(f"Model Test RMSE: {rmse:.4f}")

print(f"Baseline R2: {baseline_r2:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")

Model Test R2: 0.1349
Model Test RMSE: 0.9660
Baseline R2: -0.0002
Baseline RMSE: 1.0386


In [16]:
model_path = OUTPUT_DIR / f"distilbert_admissions_{ACTIVE_PRESET}.pt"
predictions_path = OUTPUT_DIR / f"distilbert_predictions_{ACTIVE_PRESET}.csv"
metrics_path = OUTPUT_DIR / f"distilbert_metrics_{ACTIVE_PRESET}.json"
config_path = OUTPUT_DIR / f"distilbert_config_{ACTIVE_PRESET}.json"

torch.save(model.state_dict(), model_path)

predictions_df = test_df[[
    "titleId",
    "originalTitle",
    "country",
    "admissions",
    "log_admissions",
    "admissions_zscore",
    "budget_eur",
    "budget_bracket",
    "cluster",
    "mainCountry",
    "firstLanguage",
    "genres",
]].copy()

predictions_df["prediction"] = all_preds
predictions_df["error"] = predictions_df["prediction"] - predictions_df["admissions_zscore"]

predictions_df.to_csv(predictions_path, index=False)

metrics = {
    "run_name": RUN_NAME,
    "run_number": RUN_NUMBER,
    "active_preset": ACTIVE_PRESET,
    "output_dir": str(OUTPUT_DIR),
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "model_name": CFG["model_name"],
    "freeze_encoder": CFG["freeze_encoder"],
    "max_len": CFG["max_len"],
    "batch_size": CFG["batch_size"],
    "epochs": CFG["epochs"],
    "learning_rate": CFG["lr"],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_films": int(train_df["titleId"].nunique()),
    "test_films": int(test_df["titleId"].nunique()),
    "countries": int(df_long["country"].nunique()),
    "numeric_columns": NUMERIC_COLUMNS,
    "model_r2": float(r2),
    "model_rmse": float(rmse),
    "baseline_r2": float(baseline_r2),
    "baseline_rmse": float(baseline_rmse),
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

config_to_save = {
    "run_name": RUN_NAME,
    "run_number": RUN_NUMBER,
    "active_preset": ACTIVE_PRESET,
    "preset_config": CFG,
    "random_state": RANDOM_STATE,
    "min_films_per_country": MIN_FILMS_PER_COUNTRY,
    "numeric_columns": NUMERIC_COLUMNS,
    "output_dir": str(OUTPUT_DIR),
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_to_save, f, indent=2)

print("Saved model to:", model_path)
print("Saved predictions to:", predictions_path)
print("Saved metrics to:", metrics_path)
print("Saved config to:", config_path)

Saved model to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu\standard6511\distilbert_admissions_standard.pt
Saved predictions to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu\standard6511\distilbert_predictions_standard.csv
Saved metrics to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu\standard6511\distilbert_metrics_standard.json
Saved config to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\distilbert_regressor_xpu\standard6511\distilbert_config_standard.json


In [17]:
predictions_df.sort_values("error", ascending=False).head(10)

,titleId,originalTitle,country,admissions,log_admissions,admissions_zscore,budget_eur,budget_bracket,cluster,mainCountry,firstLanguage,genres,prediction,error
2297,tt0052964,Katia,PL,6,1.945910,-4.228190,NaN,unknown,2,FR,fr,"Drama, History, Romance",-0.360379,3.867811
3358,tt0054119,Niewinni czarodzieje,PL,15,2.772589,-3.829778,NaN,unknown,2,PL,pl,"Drama, Music, Romance",-0.100483,3.729295
22522,tt2249221,Zulu,EE,33,3.526361,-3.679913,14720000.0,mid,1,FR,xh,"Action, Crime, Drama, Thriller",0.003012,3.682925
21696,tt0402901,The Cave,SI,13,2.639057,-3.495461,27600000.0,big,3,DE,en,"Adventure, Horror, Sci-Fi, Thriller",0.070768,3.566230
20303,tt0105665,Twin Peaks: Fire Walk with Me,PL,14,2.708050,-3.860882,9200000.0,mid,3,FR,en,"Drama, Horror, Mystery, Thriller",-0.337902,3.522981
16741,tt1242545,Looking for Eric,CZ,26,3.295837,-3.731737,4680000.0,european,5,GB,en,"Comedy, Drama, Fantasy, Sport",-0.304512,3.427226
21427,tt0091867,A Room with a View,PL,21,3.091042,-3.676302,2760000.0,micro,2,GB,en,"Drama, Romance",-0.304711,3.371591
2784,tt0244521,Funny Money,HU,42,3.761200,-3.553383,NaN,unknown,5,DE,en,Comedy,-0.253715,3.299669
4413,tt0241527,Harry Potter and the Sorcerer's Stone,PT,20,3.044522,-2.412949,115000000.0,big,0,GB,en,"Adventure, Family, Fantasy",0.865982,3.278930
3102,tt0057358,Nattvardsgästerna,TR,42,3.761200,-3.592647,NaN,unknown,4,SE,sv,Drama,-0.548902,3.043745


In [18]:
predictions_df.sort_values("error", ascending=True).head(10)

,titleId,originalTitle,country,admissions,log_admissions,admissions_zscore,budget_eur,budget_bracket,cluster,mainCountry,firstLanguage,genres,prediction,error
2877,tt4944610,Pet,TR,1647727,14.314908,2.274750,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.573889,-2.848638
9605,tt9320300,The Rise of the Synths,GB,7775667,15.866510,2.181943,300000.0,micro,0,ES,en,"Documentary, Music",-0.587435,-2.769378
9608,tt9320300,The Rise of the Synths,HU,719829,13.486770,2.099587,300000.0,micro,0,ES,en,"Documentary, Music",-0.600538,-2.700124
2849,tt4944610,Pet,CZ,669613,13.414457,2.031799,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.577563,-2.609361
2852,tt4944610,Pet,EE,81437,11.307597,2.027206,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.570752,-2.597958
9609,tt9320300,The Rise of the Synths,IE,482654,13.087057,1.938087,300000.0,micro,0,ES,en,"Documentary, Music",-0.599640,-2.537727
2862,tt4944610,Pet,IS,72316,11.188815,1.953259,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.578040,-2.531299
2858,tt4944610,Pet,GR,429189,12.969655,1.936233,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.576190,-2.512423
2859,tt4944610,Pet,HR,183704,12.121086,1.932354,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.567455,-2.499810
2871,tt4944610,Pet,PL,2243204,14.623416,1.881643,NaN,unknown,3,ES,en,"Crime, Horror, Thriller",-0.579554,-2.461197
